In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

JupyterDash.infer_jupyter_proxy_config()

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output, State
import plotly.express as px
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import your CRUD module
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Update with your username and password for MongoDB
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Start with ALL records
df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True)

#########################
# Pre-build MongoDB queries for filters
#########################
# Based on the Rescue Type and Preferred Breeds table in the spec

water_query = {
    "$and": [
        {"animal_type": "Dog"},
        {"breed": {"$in": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ]}},
        {"sex_upon_outcome": "Intact Female"},
        {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
    ]
}

mountain_query = {
    "$and": [
        {"animal_type": "Dog"},
        {"breed": {"$in": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ]}},
        {"sex_upon_outcome": "Intact Male"},
        {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
    ]
}

disaster_query = {
    "$and": [
        {"animal_type": "Dog"},
        {"breed": {"$in": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ]}},
        {"sex_upon_outcome": "Intact Male"},
        {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}
    ]
}

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# --- Logo and your name ---
# Change this filename if necessary to match Codio
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([

    # Top bar with logo and your name
    html.Div(
        style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'space-between'},
        children=[
            html.A(
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image.decode()),
                    style={'height': '80px', 'width': 'auto'}
                ),
                href='https://www.snhu.edu',
                target='_blank'
            ),
            html.Div([
                html.H1("Grazioso Salvare Dashboard", style={'margin-bottom': '0'}),
                html.H4("Created by Ahmad Mansour", style={'margin-top': '0'})
            ], style={'textAlign': 'right'})
        ]
    ),

    html.Hr(),

    # --- Interactive filter options (radio buttons) ---
    html.Div(
        id='filter-div',
        children=[
            html.H5("Filter by Rescue Type:"),
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                    {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                    {'label': 'Reset (All Records)', 'value': 'reset'}
                ],
                value='reset',
                labelStyle={'display': 'inline-block', 'margin-right': '15px'}
            )
        ]
    ),

    html.Hr(),

    # --- Interactive data table ---
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '5px',
            'fontFamily': 'Arial',
            'fontSize': 12
        },
        style_header={
            'fontWeight': 'bold',
            'backgroundColor': '#f2f2f2'
        }
    ),

    html.Br(),
    html.Hr(),

    # --- Chart placeholder (pie chart will be injected here by callback) ---
    html.Div(
        id='graph-id',
        style={'width': '100%', 'margin': '0 auto'}
    ),

    html.Br(),
    html.Hr(),

    # --- Map placeholder with a default map (will be updated by callback) ---
    html.Div(
        id='map-id',
        style={'width': '100%', 'margin': '0 auto'},
        children=[
            dl.Map(
                style={'width': '100%', 'height': '500px'},
                center=[30.75, -97.48],
                zoom=10,
                children=[
                    dl.TileLayer(id="base-layer-id")
                ]
            )
        ]
    )
])
    
#############################################
# Interaction Between Components / Controller
#############################################

# 1) Update table data based on selected filter
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'water':
        query = water_query
    elif filter_type == 'mountain':
        query = mountain_query
    elif filter_type == 'disaster':
        query = disaster_query
    else:
        # reset – all records
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))
    dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')


# 2) Update the pie chart based on what is visible in the table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Pie chart of breeds
    fig = px.pie(dff, names='breed', title='Breed Distribution for Selected Dogs')
    return [
        dcc.Graph(figure=fig)
    ]


# 3) Highlight selected column in the table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns or []]


# 4) Update map based on selected row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return

    dff = pd.DataFrame.from_dict(viewData)

    # default to first row if nothing selected
    if not index:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(
                position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']],
                children=[
                    dl.Tooltip(dff.iloc[row]['breed']),
                    dl.Popup([
                        html.H4("Animal Name"),
                        html.P(dff.iloc[row]['name'])
                    ])
                ]
            )
        ])
    ]


# Run app in Jupyter
app.run_server(mode='external', debug=True)

Dash app running on https://jaguarpython-famegossip-3000.codio.io/proxy/8050/
